# Car AI Project - Cleaning

#### Introduction

This notebook takes the raw dataset produced in Notebook 01 and performs systematic, reproducible cleaning.
The goal is to ensure the dataset is:
- consistent
- complete (or consistently imputed)
- free of obvious errors
- ready for EDA and feature engineering
This step is still model‑agnostic: no scaling, no encoding, no model‑specific transformations.

## 1. Libraries

In [0]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

In [0]:
# Import project configuration
import sys
import os

# Add parent directory to path to import config
sys.path.append("..")
from config import *

print("PROJECT CONFIGURATION LOADED")
print(f"\nBASE_PATH: {BASE_PATH}")
print(f"\nData Paths:")
print(f"   - RAW_DATA_PATH: {RAW_DATA_PATH}")
print(f"   - FEATURES_PATH: {FEATURES_PATH}")
print(f"   - PROCESSED_DATA_PATH: {PROCESSED_DATA_PATH}")
print(f"   - TRAIN_TEST_PATH: {TRAIN_TEST_PATH}")
print(f"   - METRICS_PATH: {METRICS_PATH}")
print(f"\nModel Path:")
print(f"   - MODEL_PATH: {MODEL_PATH}")
print(f"\nUnity Catalog:")
print(f"   - SOURCE_CSV_FILE: {SOURCE_CSV_FILE}")
print(f"   - RAW_CARS_TABLE: {RAW_CARS_TABLE}")
print(f"   - CLEANED_CARS_TABLE: {CLEANED_CARS_TABLE}")

PROJECT CONFIGURATION LOADED

BASE_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject

Data Paths:
   - RAW_DATA_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/raw
   - FEATURES_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/features
   - PROCESSED_DATA_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/processed
   - TRAIN_TEST_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/train_test
   - METRICS_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/metrics

Model Path:
   - MODEL_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/models

Unity Catalog:
   - SOURCE_CSV_FILE: /Volumes/workspace/caraiproject/caraiproject/Cars_Datasets_2025.csv
   - RAW_CARS_TABLE: workspace.caraiproject.raw_cars_data_gathered
   - CLEANED_CARS_TABLE: workspace.caraiproject.cleaned_cars_data


## 2. Load Dataset

### 2.1 Import Delta Table in a Pandas Dataframe

In [0]:
# Import the raw Delta Table into pandas DataFrame
# Use config constant for source table
source_table = RAW_CARS_TABLE

# Read Delta Table using Spark (required for Delta format)
spark_df = spark.table(source_table)
row_count_spark = spark_df.count()

# Convert to pandas DataFrame for data manipulation
df = spark_df.toPandas()

# Display dataset information
print("DATASET INFORMATION")
print(f"\nShape: {df.shape}")

print(f"\nColumns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col:30s} ({df[col].dtype})")

display(df.head())

print(f"\nDataset ready for cleaning")

DATASET INFORMATION

Shape: (1218, 11)

Columns (11):
   1. company_names                  (object)
   2. cars_names                     (object)
   3. engines                        (object)
   4. cc_battery_capacity            (object)
   5. horsepower                     (object)
   6. total_speed                    (object)
   7. performance_0_100_kmh          (object)
   8. cars_prices                    (object)
   9. fuel_types                     (object)
  10. seats                          (object)
  11. torque                         (object)


company_names,cars_names,engines,cc_battery_capacity,horsepower,total_speed,performance_0_100_kmh,cars_prices,fuel_types,seats,torque
Porsche,911 Carrera S Cabriolet,3.0L Twin-Turbo Flat-6,2981 cc,443 hp,304 km/h,3.9 sec,"$129,900",Petrol,4,530 Nm
Porsche,911 Carrera 4,3.0L Twin-Turbo Flat-6,2981 cc,379 hp,293 km/h,4.4 sec,"$113,300",Petrol,4,450 Nm
Porsche,911 Carrera 4 GTS,3.0L Twin-Turbo Flat-6,2981 cc,473 hp,311 km/h,3.3 sec,"$137,000",Petrol,4,570 Nm
Porsche,911 Targa 4 GTS,3.0L Twin-Turbo Flat-6,2981 cc,473 hp,307 km/h,3.5 sec,"$157,300",Petrol,4,570 Nm
Porsche,911 Turbo Cabriolet,3.8L Twin-Turbo Flat-6,3745 cc,572 hp,320 km/h,2.9 sec,"$184,100",Petrol,4,750 Nm



Dataset ready for cleaning


## 3 Cleaning

### 3.1 Duplicates

In [0]:
# Identify duplicate rows
duplicate_rows = df[df.duplicated(keep=False)]

if not duplicate_rows.empty:
    print(f"Duplicate rows identified: {len(duplicate_rows)}")
    # Display duplicate rows
    display(duplicate_rows)
else:
    print("No duplicate rows found.")

Duplicate rows identified: 8


company_names,cars_names,engines,cc_battery_capacity,horsepower,total_speed,performance_0_100_kmh,cars_prices,fuel_types,seats,torque
Tata Motors,Tiago EV,Permanent Magnet Synchronous,24 kWh,74 hp,120 km/h,5.7 sec,"$11,000",Electric,5,114 Nm
Tata Motors,Tiago EV,Permanent Magnet Synchronous,24 kWh,74 hp,120 km/h,5.7 sec,"$11,000",Electric,5,114 Nm
Chevrolet,Tahoe RST,5.3L V8 Gasoline,5300 cc,355 hp,180 km/h,7.4 sec,"$64,100",Petrol,7,519 Nm
Chevrolet,Tahoe RST,5.3L V8 Gasoline,5300 cc,355 hp,180 km/h,7.4 sec,"$64,100",Petrol,7,519 Nm
Volkswagen,Golf Cabriolet,1.2L I4 Turbo / 2.0L I4 Turbo,1197 cc / 1984 cc,105 - 210 hp,217 km/h,8.4 sec,"$30,000",Petrol,4,200 Nm
Volkswagen,Golf Cabriolet,1.2L I4 Turbo / 2.0L I4 Turbo,1197 cc / 1984 cc,105 - 210 hp,217 km/h,8.4 sec,"$30,000",Petrol,4,200 Nm
Volkswagen,Jetta Hybrid,1.4L I4 Turbo + Electric Motor,1395 cc,170 hp,210 km/h,8.6 sec,"$28,000",Hybrid,5,250 Nm
Volkswagen,Jetta Hybrid,1.4L I4 Turbo + Electric Motor,1395 cc,170 hp,210 km/h,8.6 sec,"$28,000",Hybrid,5,250 Nm


In [0]:
# Remove duplicate rows, keeping the first occurrence
# Full duplicates
df_cleaned = df.drop_duplicates().copy()

# Logical duplicates check (diagnostic only)
logical_duplicates = df_cleaned[df_cleaned.duplicated(
    subset=["company_names", "cars_names", "cars_prices"], 
    keep=False
)]

print(f"Logical duplicates detected: {len(logical_duplicates)}")

Logical duplicates detected: 2


In [0]:
logical_duplicates

,company_names,cars_names,engines,cc_battery_capacity,horsepower,total_speed,performance_0_100_kmh,cars_prices,fuel_types,seats,torque
214,Peugeot,508 PSE (Peugeot Sport Engineered),Hybrid I4 + Electric Motor,"1,599 cc (petrol) + Electric Motor",360 hp,250 km/h,5.2 sec,"$55,000",Hybrid,5,520 Nm
371,Peugeot,508 PSE (Peugeot Sport Engineered),Hybrid I4,"1,598 cc (petrol) + Electric Motor",360 hp,250 km/h,5.2 sec,"$55,000",Hybrid,5,520 Nm


In [0]:
# logical duplicate Peugeot 508 PSE
df_cleaned = df_cleaned.drop(index=371).copy()

### 3.2 Missing Values and Column Normalization


In [0]:
# Inspect Rows with Missing Values
# Focus on data columns (exclude metadata)
metadata_columns = ['load_timestamp', 'source_file', 'data_version']
data_columns = [col for col in df_cleaned.columns if col not in metadata_columns]

# Find rows with at least one missing value in data columns
rows_with_missing = df_cleaned[df_cleaned[data_columns].isnull().any(axis=1)]

print(f"\nTotal rows with missing values: {len(rows_with_missing)}")
print(f"Total rows in dataset: {len(df)}")
print(f"Percentage of rows with missing: {len(rows_with_missing)/len(df)*100:.2f}%")

if len(rows_with_missing) > 0:
    print(f"\nMissing values breakdown by column:")
    for col in data_columns:
        missing_in_col = rows_with_missing[col].isnull().sum()
        if missing_in_col > 0:
            print(f"  - {col:30s}: {missing_in_col} missing values")
    
    
    # Display all rows with missing values
    display(rows_with_missing)
    
else:
    print("\nNo rows with missing values found")

print("\nMissing rate per column (%):")
print((df_cleaned[data_columns].isna().mean() * 100).sort_values(ascending=False))


Total rows with missing values: 8
Total rows in dataset: 1218
Percentage of rows with missing: 0.66%

Missing values breakdown by column:
  - cc_battery_capacity           : 3 missing values
  - performance_0_100_kmh         : 6 missing values
  - torque                        : 1 missing values


company_names,cars_names,engines,cc_battery_capacity,horsepower,total_speed,performance_0_100_kmh,cars_prices,fuel_types,seats,torque
Mitsubishi,Canter (Truck),3.0L Diesel,"3,000 cc",150 hp,120 km/h,null,"$35,000 - $45,000",Diesel,2,370 Nm
Mitsubishi,Fuso Fighter (Truck),7.5L Diesel,"7,500 cc",240 hp,120 km/h,null,"$70,000 - $90,000",Diesel,2,686 Nm
Mitsubishi,Fuso Canter Eco Hybrid,3.0L Diesel Hybrid,"3,000 cc",150 hp,120 km/h,null,"$50,000 - $60,000",Diesel Hybrid,2,370 Nm
Mitsubishi,Minicab MiEV,Electric Motor,16 kWh battery,64 hp,100 km/h,null,"$18,000 - $20,000",Electric,2,196 Nm
Mazda,RX-Vision Concept,Rotary SkyActiv-R,null,400 hp (est.),300 km/h (est.),null,N/A (Concept Only),Petrol (Hybrid),2,null
MERCEDES,BENZ EQS 53,ELECTRIC MOTOR,null,751 hp,250 km/h,3.4 sec,"$102,000",Electric,5,950 Nm
TOYOTA,COASTER,"4.0L,DIESEL","4,009 cc",134 hp,120 km/h,null,"$70,000",Diesel,20,500 Nm
Nissan,Urvan,2.5L Turbo Diese,null,2488 cc,140 km/h,14.0 sec,"$28,000",Diesel,15,356 Nm



Missing rate per column (%):
performance_0_100_kmh    0.494641
cc_battery_capacity      0.247321
torque                   0.082440
company_names            0.000000
cars_names               0.000000
engines                  0.000000
horsepower               0.000000
total_speed              0.000000
cars_prices              0.000000
fuel_types               0.000000
seats                    0.000000
dtype: float64


**Note:**  
Before deciding how to handle missing values, each affected row was manually reviewed to understand whether the missing information is:

- structurally unavailable (not applicable to the vehicle type),
- deducible from other fields,
- or genuinely missing (and potentially imputable).

The dataset contains 8 rows with missing values (~0.66%), distributed across three columns:
- performance_0_100_kmh: 6 missing values
- cc_battery_capacity: 3 missing values
- torque: 1 missing value

These missing values are not random but concentrated in specific vehicle categories.

---

### • Mitsubishi (4 rows)
Models: Canter, Fuso Fighter, Fuso Canter Eco Hybrid, Minicab MiEV  
Missing: 0–100 km/h performance  

→ These are commercial vehicles (trucks / microvans).  
Acceleration performance is typically not published or not meaningful for this class.

**Action:** Keep as NaN.  
Using 0 would incorrectly imply measurable performance and would bias both linear and tree-based models.

---

### • Mazda RX-Vision Concept
Missing: multiple fields (cc_battery_capacity, performance, torque, price)

→ Concept vehicle with no production specifications or real market pricing.

**Action:** Remove from dataset (already handled in cleaning step).  
Concept cars are not valid targets for regression modeling.

---

### • Mercedes-Benz EQS 53
Missing: cc_battery_capacity

→ Fully electric vehicle; engine displacement (cc) is not applicable by design.

**Action:** Keep as NaN.  
Do not impute or set to 0, since this would introduce a physically incorrect value.

---

### • Toyota Coaster
Missing: 0-100 km/h performance

→ Minibus / passenger transport vehicle.  
Acceleration is not a standard or relevant specification.

**Action:** Keep as NaN.

---

### • Nissan Urvan
Missing: torque (1 value)

→ Torque value is structurally present but inconsistently parsed in raw data.

**Action:** This will be handled during numeric extraction consistency checks (no special imputation at this stage).

---

### • Additional structural checks
- `cc_battery_capacity` correctly contains mixed formats (cc vs kWh) and has already been split.
- No additional concept cars remain after filtering step.
- Missing values are concentrated in non-critical, vehicle-dependent attributes rather than randomly distributed.

---

**Conclusion:**  
Missing values are structurally explainable and will not be imputed arbitrarily.  
Only future feature engineering steps may introduce imputations where statistically or physically justified.

#### 3.2.1 Check for Concept Cars

In [0]:
# Define concept-related keywords
concept_keywords = ["concept", "prototype", "vision"]

# Work directly on cars_names (current state of notebook)
cars_lower = df_cleaned["cars_names"].str.lower()

# Detect concept cars using only semantic column
concept_mask = cars_lower.str.contains(
    "|".join(concept_keywords),
    na=False
)

concept_cars = df_cleaned[concept_mask]

print(f"Concept cars found: {len(concept_cars)}")
display(concept_cars)

Concept cars found: 2


company_names,cars_names,engines,cc_battery_capacity,horsepower,total_speed,performance_0_100_kmh,cars_prices,fuel_types,seats,torque
Mazda,RX-Vision Concept,Rotary SkyActiv-R,null,400 hp (est.),300 km/h (est.),null,N/A (Concept Only),Petrol (Hybrid),2,null
Mazda,MX-5 Superlight Concept,1.8L Inline-4,"1,798 cc",143 hp,200 km/h,7.5 sec,"$30,000 - $35,000",Petrol,2,160 Nm


**Note**: A search was performed for concept models using keywords such as 
**"concept"**, **"prototype"**, and **"vision"** in `cars_names`.

Two concept cars were identified:
- Mazda RX-Vision Concept  
- Mazda MX-5 Superlight Concept  

Concept cars do not have real production prices or finalized specifications, so they cannot be used
as valid targets for a regression model.

In [0]:
# Remove all cars found in concept_cars from df_cleaned
df_cleaned = df_cleaned[~df_cleaned.index.isin(concept_cars.index)]
print(f"Rows after removing concept cars: {df_cleaned.shape[0]}")
display(df_cleaned.head())

Rows after removing concept cars: 1211


company_names,cars_names,engines,cc_battery_capacity,horsepower,total_speed,performance_0_100_kmh,cars_prices,fuel_types,seats,torque
Porsche,911 Carrera S Cabriolet,3.0L Twin-Turbo Flat-6,2981 cc,443 hp,304 km/h,3.9 sec,"$129,900",Petrol,4,530 Nm
Porsche,911 Carrera 4,3.0L Twin-Turbo Flat-6,2981 cc,379 hp,293 km/h,4.4 sec,"$113,300",Petrol,4,450 Nm
Porsche,911 Carrera 4 GTS,3.0L Twin-Turbo Flat-6,2981 cc,473 hp,311 km/h,3.3 sec,"$137,000",Petrol,4,570 Nm
Porsche,911 Targa 4 GTS,3.0L Twin-Turbo Flat-6,2981 cc,473 hp,307 km/h,3.5 sec,"$157,300",Petrol,4,570 Nm
Porsche,911 Turbo Cabriolet,3.8L Twin-Turbo Flat-6,3745 cc,572 hp,320 km/h,2.9 sec,"$184,100",Petrol,4,750 Nm


**Note**: Concept cars were removed.

#### 3.2.2 Data Quality - Invalid Records

In [0]:
# Identify records with clearly invalid horsepower values
# Focus on extremely high values that are physically impossible for standard vehicles

# Extract numeric horsepower for analysis
df_cleaned['hp_numeric'] = df_cleaned['horsepower'].str.extract(r'(\d+)').astype(float)

# Check for outliers (hp > 2000 is unrealistic for most production vehicles)
suspect_hp = df_cleaned[df_cleaned['hp_numeric'] > 2000]

print(f"Records with suspicious horsepower (>2000 hp): {len(suspect_hp)}")
if len(suspect_hp) > 0:
    display(suspect_hp[['company_names', 'cars_names', 'horsepower', 'hp_numeric', 'engines', 'cars_prices']])

# Drop temporary column
df_cleaned = df_cleaned.drop(columns=['hp_numeric'])

Records with suspicious horsepower (>2000 hp): 1


company_names,cars_names,horsepower,hp_numeric,engines,cars_prices
Nissan,Urvan,2488 cc,2488.0,2.5L Turbo Diese,"$28,000"


In [0]:
# Remove Nissan Urvan with invalid horsepower (2488 hp)
# This is clearly a data entry error (likely torque value recorded as horsepower)

rows_before = len(df_cleaned)

# Find the problematic record
invalid_mask = (
    (df_cleaned['company_names'].str.contains('Nissan', case=False, na=False)) &
    (df_cleaned['cars_names'].str.contains('Urvan', case=False, na=False)) &
    (df_cleaned['horsepower'].str.contains('2488', na=False))
)

print(f"Nissan Urvan records with 2488 hp found: {invalid_mask.sum()}")

if invalid_mask.sum() > 0:
    print("\nRecord to be removed:")
    display(df_cleaned[invalid_mask])
    
    # Remove the invalid record
    df_cleaned = df_cleaned[~invalid_mask].copy()
    
    print(f"\n✅ Record removed")
    print(f"Rows before: {rows_before}")
    print(f"Rows after: {len(df_cleaned)}")
else:
    print("No matching record found (may have been removed already)")

Nissan Urvan records with 2488 hp found: 1

Record to be removed:


company_names,cars_names,engines,cc_battery_capacity,horsepower,total_speed,performance_0_100_kmh,cars_prices,fuel_types,seats,torque
Nissan,Urvan,2.5L Turbo Diese,null,2488 cc,140 km/h,14.0 sec,"$28,000",Diesel,15,356 Nm



✅ Record removed
Rows before: 1211
Rows after: 1210


**Note:**  
A data quality check identified a **Nissan Urvan** record with **2488 hp**, which is physically impossible for a commercial minivan.

**Analysis:**
- Typical Nissan Urvan models have ~130-160 hp
- The value 2488 likely represents **torque (Nm)** incorrectly recorded as horsepower
- This is a clear data entry error that would severely distort regression models

**Action:** Record removed to ensure data quality and model reliability.

#### 3.2.3 cc_battery_capacity

**Note**:  
The column `cc_battery_capacity` mixes two different concepts:
- engine displacement (cc) for combustion vehicles
- battery capacity (kWh) for electric vehicles

In [0]:
### 3.2.3 cc_battery_capacity

# Initialize structured columns
df_cleaned["engine_cc"] = np.nan
df_cleaned["battery_kwh"] = np.nan

# Work on raw column
cc_raw = df_cleaned["cc_battery_capacity"].astype(str)

# Normalize text
cc_clean = (
    cc_raw.str.lower()
    .str.strip()
)

# HYBRID PATTERN
# e.g. "1599 cc (petrol) + Electric Motor"
hybrid_mask = cc_clean.str.contains(r"\+.*electric", na=False)

df_cleaned.loc[hybrid_mask, "engine_cc"] = (
    cc_clean[hybrid_mask]
    .str.extract(r"(\d+\.?\d*)", expand=False)
    .astype(float)
)

# battery still unknown in hybrids → keep NaN
df_cleaned.loc[hybrid_mask, "battery_kwh"] = np.nan

# ELECTRIC PATTERN
# e.g. "16 kWh battery"
ev_mask = cc_clean.str.contains("kwh", na=False)

df_cleaned.loc[ev_mask, "battery_kwh"] = (
    cc_clean[ev_mask]
    .str.extract(r"(\d+\.?\d*)", expand=False)
    .astype(float)
)

# EVs have no displacement
df_cleaned.loc[ev_mask, "engine_cc"] = np.nan

# ICE ENGINE PATTERN
# e.g. "3,000 cc"
ice_mask = cc_clean.str.contains("cc", na=False) & ~hybrid_mask

df_cleaned.loc[ice_mask, "engine_cc"] = (
    cc_clean[ice_mask]
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+\.?\d*)", expand=False)
    .astype(float)
)

# AMBIGUOUS / MULTI-VALUE PATTERN
multi_mask = cc_clean.str.contains(r"/|,.*cc.*cc", na=False)

df_cleaned.loc[multi_mask, ["engine_cc", "battery_kwh"]] = np.nan

# Drop original column (after extraction)
df_cleaned = df_cleaned.drop(columns=["cc_battery_capacity"])

df_cleaned[["engine_cc", "battery_kwh"]].head()

,engine_cc,battery_kwh
0,2981.0,NaN
1,2981.0,NaN
2,2981.0,NaN
3,2981.0,NaN
4,3745.0,NaN


In [0]:
df_cleaned.head()

,company_names,cars_names,engines,horsepower,total_speed,performance_0_100_kmh,cars_prices,fuel_types,seats,torque,engine_cc,battery_kwh
0,Porsche,911 Carrera S Cabriolet,3.0L Twin-Turbo Flat-6,443 hp,304 km/h,3.9 sec,"$129,900",Petrol,4,530 Nm,2981.0,NaN
1,Porsche,911 Carrera 4,3.0L Twin-Turbo Flat-6,379 hp,293 km/h,4.4 sec,"$113,300",Petrol,4,450 Nm,2981.0,NaN
2,Porsche,911 Carrera 4 GTS,3.0L Twin-Turbo Flat-6,473 hp,311 km/h,3.3 sec,"$137,000",Petrol,4,570 Nm,2981.0,NaN
3,Porsche,911 Targa 4 GTS,3.0L Twin-Turbo Flat-6,473 hp,307 km/h,3.5 sec,"$157,300",Petrol,4,570 Nm,2981.0,NaN
4,Porsche,911 Turbo Cabriolet,3.8L Twin-Turbo Flat-6,572 hp,320 km/h,2.9 sec,"$184,100",Petrol,4,750 Nm,3745.0,NaN


**Note**: After extracting clean numeric fields (`engine_cc` and `battery_kwh`) and handling hybrids, multi-value entries, and non-numeric patterns, the original `cc_battery_capacity` column no longer provides useful information and contains unstructured text. It was therefore removed to avoid redundancy and noise.

**Note:**  
The `cc_battery_capacity` column contains mixed powertrain information and has been split into:

- `engine_cc` → internal combustion engine displacement (cc)
- `battery_kwh` → electric battery capacity (kWh)

Three structural cases are handled:

- **ICE vehicles** → only engine displacement is extracted  
- **Electric vehicles** → only battery capacity is extracted  
- **Hybrid vehicles** → engine displacement is extracted, battery remains unknown  

Entries with ambiguous or multi-value formats are not imputed to avoid introducing
physically incorrect assumptions.

This approach preserves the physical meaning of the variables and avoids artificial bias
in downstream modeling.

#### 3.2.3 performance_0_100kwh

In [0]:
# 'performance_0_100_kmh'
grouped_perf = df_cleaned.groupby('performance_0_100_kmh').size().reset_index(name='count')
display(grouped_perf)

performance_0_100_kmh,count
6.1 sec,1
1.9 sec,1
10 sec,2
10. 5 sec,1
10.0 sec,19
10.1 sec,1
10.2 sec,14
10.3 sec,5
10.4 sec,3
10.5 sec,38


**Note**:The performance field contains many heterogeneous formats. Only a single, valid
numeric 0-100 km/h value can be used. Values are cleaned as follows:


In [0]:
perf_raw = df_cleaned["performance_0_100_kmh"].copy()

perf = (
    perf_raw.str.lower()
    .str.replace("sec", "", regex=False)
    .str.strip()
)

# Initialize columns
df_cleaned["acc_min"] = np.nan
df_cleaned["acc_max"] = np.nan
df_cleaned["acc_mean"] = np.nan
df_cleaned["acc_is_range"] = 0
df_cleaned["acc_is_instant"] = 0

def extract_acc(value):
    if pd.isna(value):
        return np.nan, np.nan, 0

    v = str(value).lower()

    # instant case
    if "instant" in v:
        return 0.5, 0.5, 1

    # normalize separators for ranges
    v = v.replace("–", "-").replace("—", "-")

    # CASE 1: multi-value (NOT range)
    if "/" in v:
        nums = re.findall(r"\d+\.?\d*", v)
        if len(nums) >= 2:
            nums = list(map(float, nums))
            return min(nums), max(nums), 0

    # CASE 2: range
    if "-" in v:
        nums = re.findall(r"\d+\.?\d*", v)
        if len(nums) >= 2:
            nums = list(map(float, nums))
            return min(nums), max(nums), 0

    # CASE 3: single value (handles messy formats like "5. 0 sec", "10 sec", "4")
    nums = re.findall(r"\d+\.?\d*", v)

    if len(nums) == 0:
        return np.nan, np.nan, 0

    val = float(nums[0])
    return val, val, 0

# Apply
results = perf.apply(extract_acc)

df_cleaned["acc_min"] = results.apply(lambda x: x[0])
df_cleaned["acc_max"] = results.apply(lambda x: x[1])
df_cleaned["acc_is_instant"] = results.apply(lambda x: x[2])

df_cleaned["acc_mean"] = df_cleaned[["acc_min", "acc_max"]].mean(axis=1)
df_cleaned["acc_is_range"] = (df_cleaned["acc_min"] != df_cleaned["acc_max"]).astype(int)

# Drop original
df_cleaned = df_cleaned.drop(columns=["performance_0_100_kmh"])

In [0]:
df_cleaned[["acc_mean", "acc_is_range", "acc_is_instant"]].describe()

,acc_mean,acc_is_range,acc_is_instant
count,1205.000000,1210.000000,1210.0
mean,7.563195,0.009917,0.0
std,3.313276,0.099132,0.0
min,1.900000,0.000000,0.0
25%,5.200000,0.000000,0.0
50%,7.100000,0.000000,0.0
75%,9.500000,0.000000,0.0
max,35.000000,1.000000,0.0


In [0]:
df_cleaned["is_commercial"] = df_cleaned["cars_names"].str.contains(
    "truck|van|minibus|coaster|urvan|fighter|canter",
    case=False,
    na=False
).astype(int)


In [0]:
df_cleaned.head()

,company_names,cars_names,engines,horsepower,total_speed,cars_prices,fuel_types,seats,torque,engine_cc,battery_kwh,acc_min,acc_max,acc_mean,acc_is_range,acc_is_instant,is_commercial
0,Porsche,911 Carrera S Cabriolet,3.0L Twin-Turbo Flat-6,443 hp,304 km/h,"$129,900",Petrol,4,530 Nm,2981.0,NaN,3.9,3.9,3.9,0,0,0
1,Porsche,911 Carrera 4,3.0L Twin-Turbo Flat-6,379 hp,293 km/h,"$113,300",Petrol,4,450 Nm,2981.0,NaN,4.4,4.4,4.4,0,0,0
2,Porsche,911 Carrera 4 GTS,3.0L Twin-Turbo Flat-6,473 hp,311 km/h,"$137,000",Petrol,4,570 Nm,2981.0,NaN,3.3,3.3,3.3,0,0,0
3,Porsche,911 Targa 4 GTS,3.0L Twin-Turbo Flat-6,473 hp,307 km/h,"$157,300",Petrol,4,570 Nm,2981.0,NaN,3.5,3.5,3.5,0,0,0
4,Porsche,911 Turbo Cabriolet,3.8L Twin-Turbo Flat-6,572 hp,320 km/h,"$184,100",Petrol,4,750 Nm,3745.0,NaN,2.9,2.9,2.9,0,0,0


In [0]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1210 entries, 0 to 1217
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   company_names   1210 non-null   object 
 1   cars_names      1210 non-null   object 
 2   engines         1210 non-null   object 
 3   horsepower      1210 non-null   object 
 4   total_speed     1210 non-null   object 
 5   cars_prices     1210 non-null   object 
 6   fuel_types      1210 non-null   object 
 7   seats           1210 non-null   object 
 8   torque          1210 non-null   object 
 9   engine_cc       1062 non-null   float64
 10  battery_kwh     104 non-null    float64
 11  acc_min         1205 non-null   float64
 12  acc_max         1205 non-null   float64
 13  acc_mean        1205 non-null   float64
 14  acc_is_range    1210 non-null   int64  
 15  acc_is_instant  1210 non-null   int64  
 16  is_commercial   1210 non-null   int64  
dtypes: float64(5), int64(3), object(9)
mem

#### 3.2.4 company_names

In [0]:
# Rename the original company_names column to make (standard automotive naming)
df_cleaned = df_cleaned.rename(columns={"company_names": "make"})

In [0]:
df_cleaned['make'].groupby(df_cleaned['make']).count().sort_index()


make
ASTON MARTIN          11
AUDI                  21
Acura                 27
BENTLEY                1
BMW                   41
Bugatti               10
Cadillac              20
Chevrolet             57
FERRARI                9
Ford                  54
GMC                   59
HONDA                  7
HYUNDAI               23
Jaguar Land Rover     40
Jeep                  19
KIA                   16
KIA                    1
Kia                   54
LAMBORGHINI           24
MAHINDRA               3
MARUTI SUZUKI          3
MERCEDES              21
Mazda                 85
Mitsubishi            60
NISSAN                10
Nissan               148
Peugeot               53
Porsche               96
ROLLS ROYCE           29
ROLLS ROYCE            1
TOYOTA                45
Tata Motors           28
Tesla                 10
Toyota                 5
VOLVO                  3
Volkswagen           107
Volvo                  9
Name: make, dtype: int64

**Note**: Before cleaning the `make` column (originally `company_names`), is useful to inspect all
unique manufacturer names to identify inconsistencies. The dataset contains multiple variants of the same brand due to differences in capitalization, spacing, symbols, and formatting (e.g., "TOYOTA", "Toyota", "Mercedes", "Mercedes Benz", "ROLLS ROYCE"). These inconsistencies must be fixed before analysis or modeling.

The next step is to standardize all manufacturer names into a single, clean, consistent format. 

In [0]:
# Normalize manufacturer names

# Strip leading/trailing spaces and convert to lowercase
df_cleaned["make"] = (
    df_cleaned["make"]
    .str.strip()
    .str.lower()
)

# Replace hyphens/underscores with spaces (avoid artificial variants)
df_cleaned["make"] = (
    df_cleaned["make"]
    .str.replace("-", " ", regex=False)
    .str.replace("_", " ", regex=False)
)

# Collapse multiple spaces into a single space
df_cleaned["make"] = df_cleaned["make"].str.replace(r"\s+", " ", regex=True)

# Convert to title case (first letter uppercase for each word)
df_cleaned["make"] = df_cleaned["make"].str.title()


In [0]:
# map the correct names
make_mapping = {
    "Mercedes": "Mercedes-Benz",
    "Mercedes Benz": "Mercedes-Benz",

    "Rolls Royce": "Rolls-Royce",

    "Gmc": "GMC",
    "Bmw": "BMW",
}
# replace the names in the dataframe
df_cleaned["make"] = df_cleaned["make"].replace(make_mapping)

In [0]:
df_cleaned['make'].groupby(df_cleaned['make']).count().sort_index()

make
Acura                 27
Aston Martin          11
Audi                  21
BMW                   41
Bentley                1
Bugatti               10
Cadillac              20
Chevrolet             57
Ferrari                9
Ford                  54
GMC                   59
Honda                  7
Hyundai               23
Jaguar Land Rover     40
Jeep                  19
Kia                   71
Lamborghini           24
Mahindra               3
Maruti Suzuki          3
Mazda                 85
Mercedes-Benz         21
Mitsubishi            60
Nissan               158
Peugeot               53
Porsche               96
Rolls-Royce           30
Tata Motors           28
Tesla                 10
Toyota                50
Volkswagen           107
Volvo                 12
Name: make, dtype: int64

**Note:**  
The `company_names` column (renamed to `make`) is normalized to ensure consistent manufacturer naming.

Cleaning steps include:
- trimming whitespace
- standardizing case
- removing separators (-, _)
- collapsing multiple spaces
- applying title-case formatting
- manual correction of known brand variants

This ensures that all manufacturer names are consistent for grouping,
aggregation, and downstream feature engineering.

#### 3.2.5 cars_names

In [0]:

df_cleaned['cars_names'].head(20)


0             911 Carrera S Cabriolet
1                       911 Carrera 4
2                   911 Carrera 4 GTS
3                     911 Targa 4 GTS
4                 911 Turbo Cabriolet
5                   911 Turbo S Coupe
6                     911 GT3 Touring
7                911 GT3 RS Clubsport
8           911 Dakar Limited Edition
9         911 Carrera GTS Lightweight
10            911 Targa 4 Lightweight
11         911 Turbo Exclusive Series
12                         911 GT2 RS
13                        911 GT3 Cup
14                         911 GT4 RS
15       911 RSR (Motorsport Edition)
16                  911 Sport Turismo
17    911 Turbo Cabriolet Lightweight
18                    911 Dakar Sport
19                              Tiago
Name: cars_names, dtype: object

In [0]:
# Rename cars_names to model (standard automotive naming)
df_cleaned = df_cleaned.rename(columns={"cars_names": "model"})

In [0]:
# Inspect sample
df_cleaned["model"].head(20)

0             911 Carrera S Cabriolet
1                       911 Carrera 4
2                   911 Carrera 4 GTS
3                     911 Targa 4 GTS
4                 911 Turbo Cabriolet
5                   911 Turbo S Coupe
6                     911 GT3 Touring
7                911 GT3 RS Clubsport
8           911 Dakar Limited Edition
9         911 Carrera GTS Lightweight
10            911 Targa 4 Lightweight
11         911 Turbo Exclusive Series
12                         911 GT2 RS
13                        911 GT3 Cup
14                         911 GT4 RS
15       911 RSR (Motorsport Edition)
16                  911 Sport Turismo
17    911 Turbo Cabriolet Lightweight
18                    911 Dakar Sport
19                              Tiago
Name: model, dtype: object

In [0]:
# Basic normalization of model names
# Strip leading/trailing spaces
df_cleaned["model"] = df_cleaned["model"].str.strip()

# Replace multiple spaces with a single space
df_cleaned["model"] = df_cleaned["model"].str.replace(r"\s+", " ", regex=True)

# Remove trailing punctuation (rare but possible)
df_cleaned["model"] = df_cleaned["model"].str.replace(r"[.,;:]+$", "", regex=True)

# Standardize parentheses spacing
df_cleaned["model"] = df_cleaned["model"].str.replace(r"\s*\(\s*", " (", regex=True)
df_cleaned["model"] = df_cleaned["model"].str.replace(r"\s*\)\s*", ")", regex=True)


In [0]:
df_cleaned['model'].head(20)

0             911 Carrera S Cabriolet
1                       911 Carrera 4
2                   911 Carrera 4 GTS
3                     911 Targa 4 GTS
4                 911 Turbo Cabriolet
5                   911 Turbo S Coupe
6                     911 GT3 Touring
7                911 GT3 RS Clubsport
8           911 Dakar Limited Edition
9         911 Carrera GTS Lightweight
10            911 Targa 4 Lightweight
11         911 Turbo Exclusive Series
12                         911 GT2 RS
13                        911 GT3 Cup
14                         911 GT4 RS
15       911 RSR (Motorsport Edition)
16                  911 Sport Turismo
17    911 Turbo Cabriolet Lightweight
18                    911 Dakar Sport
19                              Tiago
Name: model, dtype: object

**Note:**  
The `cars_names` column is renamed to `model` and cleaned using basic normalization rules.

The goal of this step is to standardize formatting without altering the semantic content.

Cleaning includes:
- trimming whitespace
- collapsing multiple spaces
- normalizing dash characters
- standardizing parentheses spacing
- removing trailing punctuation

Full model names (including trims, variants, and special editions) are preserved,
as they contain important information for price prediction and will be processed
in later feature engineering steps.

#### 3.2.6 engines

In [0]:
df_cleaned['engines'].groupby(df_cleaned['engines']).count().sort_index()

engines
0.36L Inline-2                  1
0.66L Inline-3                  1
0.7L 3-Cylinder                 5
1.0L EcoBoost / 1.5L            1
1.0L EcoBoost / 1.5L / 2.0L     1
                               ..
V6 Petrol                       4
V6 Turbocharged                 1
V8                             34
V8 Gasoline Engine              2
V8 Petrol                       8
Name: engines, Length: 354, dtype: int64

**Note**: The `engines` column contains heterogeneous textual descriptions of the powertrain.
These entries may include engine displacement (in liters or cc), cylinder layout (e.g., Inline-4, V6, V8), aspiration (Turbo, Naturally Aspirated), fuel type, or electric motor configurations.

Because the formatting is inconsistent (extra spaces, mixed separators, variable use of parentheses, inconsistent unit notation), this step focuses solely on standardizing the text:
- remove extra spaces,
- normalize separators and parentheses,
- ensure consistent formatting of units and symbols.


In [0]:
# Basic cleaning of the engines column (formatting only)

# Strip leading/trailing spaces
df_cleaned["engines"] = df_cleaned["engines"].str.strip()

# Replace multiple spaces with a single space
df_cleaned["engines"] = df_cleaned["engines"].str.replace(r"\s+", " ", regex=True)

# Fix common typos (minimal cleaning)
df_cleaned["engines"] = df_cleaned["engines"].str.replace("Diese", "Diesel", case=False)

# Normalize spacing around slashes
df_cleaned["engines"] = df_cleaned["engines"].str.replace(r"\s*/\s*", " / ", regex=True)

# Normalize parentheses spacing
df_cleaned["engines"] = df_cleaned["engines"].str.replace(r"\s*\(\s*", " (", regex=True)
df_cleaned["engines"] = df_cleaned["engines"].str.replace(r"\s*\)\s*", ")", regex=True)

# Standardize unit notation (L, cc)
df_cleaned["engines"] = df_cleaned["engines"].str.replace(" l", "L", case=False)
df_cleaned["engines"] = df_cleaned["engines"].str.replace(r"\bcc\b", "cc", case=False, regex=True)

# Remove trailing punctuation if present
df_cleaned["engines"] = df_cleaned["engines"].str.replace(r"[.,;:]+$", "", regex=True)


In [0]:
df_cleaned['engines'].groupby(df_cleaned['engines']).count().sort_index()

engines
0.36L Inline-2                  1
0.66L Inline-3                  1
0.7L 3-Cylinder                 5
1.0L EcoBoost / 1.5L            1
1.0L EcoBoost / 1.5L / 2.0L     1
                               ..
V6 Petrol                       4
V6 Turbocharged                 1
V8                             34
V8 Gasoline Engine              2
V8 Petrol                       8
Name: engines, Length: 353, dtype: int64

#### 3.2.7 horsepower

In [0]:
df_cleaned

,make,model,engines,horsepower,total_speed,cars_prices,fuel_types,seats,torque,engine_cc,battery_kwh,acc_min,acc_max,acc_mean,acc_is_range,acc_is_instant,is_commercial
0,Porsche,911 Carrera S Cabriolet,3.0L Twin-Turbo Flat-6,443 hp,304 km/h,"$129,900",Petrol,4,530 Nm,2981.0,NaN,3.9,3.9,3.9,0,0,0
1,Porsche,911 Carrera 4,3.0L Twin-Turbo Flat-6,379 hp,293 km/h,"$113,300",Petrol,4,450 Nm,2981.0,NaN,4.4,4.4,4.4,0,0,0
2,Porsche,911 Carrera 4 GTS,3.0L Twin-Turbo Flat-6,473 hp,311 km/h,"$137,000",Petrol,4,570 Nm,2981.0,NaN,3.3,3.3,3.3,0,0,0
3,Porsche,911 Targa 4 GTS,3.0L Twin-Turbo Flat-6,473 hp,307 km/h,"$157,300",Petrol,4,570 Nm,2981.0,NaN,3.5,3.5,3.5,0,0,0
4,Porsche,911 Turbo Cabriolet,3.8L Twin-Turbo Flat-6,572 hp,320 km/h,"$184,100",Petrol,4,750 Nm,3745.0,NaN,2.9,2.9,2.9,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1213,Porsche,Taycan Cross Turismo GTS,Dual Electric Motors,590 hp,250 km/h,"$139,400",Electric,4,620 Nm,NaN,93.4,3.8,3.8,3.8,0,0,0
1214,Porsche,Taycan Sport Turismo Base,Dual Electric Motors,375 hp,230 km/h,"$90,500",Electric,4,500 Nm,NaN,79.2,5.4,5.4,5.4,0,0,0
1215,Porsche,Taycan Sport Turismo GTS,Dual Electric Motors,590 hp,250 km/h,"$136,700",Electric,4,620 Nm,NaN,93.4,3.7,3.7,3.7,0,0,0
1216,Porsche,Taycan Turbo S Cross Turismo,Dual Electric Motors,750 hp,260 km/h,"$190,000",Electric,4,1050 Nm,NaN,93.4,2.9,2.9,2.9,0,0,0


In [0]:
df_cleaned['horsepower'].groupby(df_cleaned['horsepower']).count().sort_index()
# Basic cleaning of the horsepower column (formatting only)


horsepower
1,020 hp          2
1,288 hp          1
100 - 150 hp      1
100 HP            3
100 hp            7
                 ..
99 hp             1
Up to 1,000 hp    2
Up to 754 hp      1
Up to 830 hp      3
~170 hp           1
Name: horsepower, Length: 454, dtype: int64

**Note**: 
The `horsepower` column contains textual values with inconsistent formats:
- numeric values with units (e.g., "120 hp", "1,020 hp"),
- uppercase/lowercase variations ("HP" vs "hp"),
- ranges ("100-150 hp"),
- approximate values ("~170 hp"),
- upper-bound expressions ("Up to 830 hp"),
- multiple values ("118 hp / 110 hp").


In [0]:
hp_raw = df_cleaned["horsepower"].copy()

# Clean base string
hp = (
    hp_raw.str.lower()
    .str.replace("hp", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace("~", "", regex=False)
    .str.replace("up to", "", regex=False)
    .str.replace("(est.)", "", regex=False)
    .str.replace(r"[()]", "", regex=True)
    .str.strip()
)
# Initialize columns
df_cleaned["hp_min"] = np.nan
df_cleaned["hp_max"] = np.nan
df_cleaned["hp_mean"] = np.nan
df_cleaned["hp_is_range"] = 0

# Extract numbers
def extract_hp_values(val):
    if pd.isna(val):
        return np.nan, np.nan
    
    nums = re.findall(r"\d+\.?\d*", val)
    
    if not nums:
        return np.nan, np.nan
    
    nums = list(map(float, nums))
    
    if len(nums) == 1:
        return nums[0], nums[0]
    
    return min(nums), max(nums)

# Apply extraction
hp_values = hp.apply(extract_hp_values)

df_cleaned["hp_min"] = hp_values.apply(lambda x: x[0])
df_cleaned["hp_max"] = hp_values.apply(lambda x: x[1])

# Mean
df_cleaned["hp_mean"] = df_cleaned[["hp_min", "hp_max"]].mean(axis=1)

# Range flag
df_cleaned["hp_is_range"] = (df_cleaned["hp_min"] != df_cleaned["hp_max"]).astype(int)

In [0]:
# Drop original column
df_cleaned = df_cleaned.drop(columns=["horsepower"])

In [0]:
df_cleaned[df_cleaned["hp_is_range"] == 1].shape[0]

89

In [0]:
df_cleaned.head()

,make,model,engines,total_speed,cars_prices,fuel_types,seats,torque,engine_cc,battery_kwh,acc_min,acc_max,acc_mean,acc_is_range,acc_is_instant,is_commercial,hp_min,hp_max,hp_mean,hp_is_range
0,Porsche,911 Carrera S Cabriolet,3.0L Twin-Turbo Flat-6,304 km/h,"$129,900",Petrol,4,530 Nm,2981.0,NaN,3.9,3.9,3.9,0,0,0,443.0,443.0,443.0,0
1,Porsche,911 Carrera 4,3.0L Twin-Turbo Flat-6,293 km/h,"$113,300",Petrol,4,450 Nm,2981.0,NaN,4.4,4.4,4.4,0,0,0,379.0,379.0,379.0,0
2,Porsche,911 Carrera 4 GTS,3.0L Twin-Turbo Flat-6,311 km/h,"$137,000",Petrol,4,570 Nm,2981.0,NaN,3.3,3.3,3.3,0,0,0,473.0,473.0,473.0,0
3,Porsche,911 Targa 4 GTS,3.0L Twin-Turbo Flat-6,307 km/h,"$157,300",Petrol,4,570 Nm,2981.0,NaN,3.5,3.5,3.5,0,0,0,473.0,473.0,473.0,0
4,Porsche,911 Turbo Cabriolet,3.8L Twin-Turbo Flat-6,320 km/h,"$184,100",Petrol,4,750 Nm,3745.0,NaN,2.9,2.9,2.9,0,0,0,572.0,572.0,572.0,0


#### 3.2.8 total_speed

In [0]:
df_cleaned['total_speed'].groupby(df_cleaned['total_speed']).count().head(50)

total_speed
100 km/h      2
105 km/h      1
110 km/h      4
120 km/h     14
125 km/h      1
130 km/h     15
138 km/h      1
140 km/h     15
145 km/h      9
148 km/h      1
150 km/h     10
155 km/h      5
160 km/h     44
162 km/h      1
165 km/h     12
168 km/h      1
170 km/h     47
175 km/h     24
177 km/h      7
180 km/h    128
183 km/h      1
185 km/h     18
186 km/h      2
187 km/h      1
190 km/h     92
191 km/h      2
193 km/h      1
195 km/h      8
196 km/h      1
200 km/h    138
201 km/h      1
204 km/h      1
205 km/h      9
209 km/h      5
210 km/h     84
211 km/h      1
213 km/h      5
215 km/h      8
216 km/h      2
217 km/h      3
220 km/h     48
221 km/h      1
222 km/h      1
225 km/h     12
226 km/h      2
230 km/h     24
232 km/h      3
233 km/h      2
235 km/h     10
237 km/h      1
Name: total_speed, dtype: int64

**Note**: The `total_speed` column contains top‑speed values expressed as text in the
format "<number> km/h". The formatting is generally consistent, but the column still requires basic normalization:

- remove extra spaces,
- standardize unit notation ("km/h"),
- ensure a consistent "<number> km/h" structure,
- prepare the field for numeric extraction.

No semantic decisions are needed here, since top speed is always a single deterministic value for each vehicle. Ranges or approximate values do not appear in this dataset.

In [0]:
# Basic formatting normalization
ts = df_cleaned["total_speed"].copy()

# Strip leading/trailing spaces
ts = ts.str.strip()

# Normalize multiple spaces
ts = ts.str.replace(r"\s+", " ", regex=True)

# Lowercase for consistency
ts = ts.str.lower()

# Standardize unit notation
ts = ts.str.replace("km/h", "km/h", regex=False)

# Remove any text after the number (e.g. "(est.)")
ts = ts.str.replace(r"\(.*?\)", "", regex=True)

# Extract numeric value (support decimals just in case)
df_cleaned["total_speed_kmh"] = (
    ts.str.extract(r"(\d+\.?\d*)", expand=False).astype(float)
)


In [0]:
df_cleaned["total_speed_kmh"].head(20)

0     304.0
1     293.0
2     311.0
3     307.0
4     320.0
5     330.0
6     318.0
7     296.0
8     240.0
9     311.0
10    289.0
11    330.0
12    340.0
13    290.0
14    315.0
15    296.0
16    293.0
17    318.0
18    240.0
19    170.0
Name: total_speed_kmh, dtype: float64

In [0]:
# Drop original column
df_cleaned = df_cleaned.drop(columns=["total_speed"])

In [0]:
# Check
df_cleaned["total_speed_kmh"].describe()

count    1210.000000
mean      216.561983
std        53.032411
min        80.000000
25%       180.000000
50%       200.000000
75%       250.000000
max       500.000000
Name: total_speed_kmh, dtype: float64

#### 3.2.9 fuel_types

In [0]:
df_cleaned['fuel_types'].groupby(df_cleaned['fuel_types']).count()

fuel_types
CNG/Petrol                   1
Diesel                     105
Diesel Hybrid                1
Diesel/Petrol                4
Electric                    96
Gas / Hybrid                 1
Hybrid                      77
Hybrid (Gas + Electric)      1
Hybrid (Petrol)              4
Hybrid / Plug-in             1
Hybrid/Electric              1
Hybrid/Petrol                1
Hydrogen                     3
Petrol                     868
Petrol, Diesel               3
Petrol, Hybrid               2
Petrol/AWD                   2
Petrol/Diesel               16
Petrol/EV                    1
Petrol/Hybrid               16
Plug-in Hybrid               5
plug in hyrbrid              1
Name: fuel_types, dtype: int64

**Note**: The `fuel_types` column contains heterogeneous labels describing the vehicle's
power source. The values show inconsistencies in separators ("/", ","), token
order, spacing, capitalization, and occasional typos (e.g., "hyrbrid"). Some
entries also include non-alphanumeric symbols (e.g., "+").

This cleaning step applies a general, rule-based normalization:

- remove non-alphanumeric symbols,
- normalize separators and spacing,
- correct common orthographic typos,
- merge multi-word expressions into consistent tokens (e.g., "plug in" → "plugin"),
- tokenize each label and sort tokens alphabetically,
- rebuild a standardized, readable label (e.g., "Hybrid Plug-In").

No semantic interpretation or category mapping is performed. The goal is to
produce a clean, consistent textual representation of each fuel type, suitable
for later feature engineering.

In [0]:

def normalize_fuel_type(value):
    if pd.isna(value):
        return value
    
    v = value.lower().strip()
    
    # Fix common typos
    v = v.replace("hyrbrid", "hybrid")
    v = v.replace("hybrd", "hybrid")
    
    # Normalize "plug in" → "plugin"
    v = v.replace("plug in", "plugin")
    v = v.replace("plug-in", "plugin")
    
    # Replace separators with space
    v = re.sub(r"[\/,;+]", " ", v)
    
    # Remove parentheses
    v = re.sub(r"[\(\)]", " ", v)
    
    # Normalize spacing
    v = re.sub(r"\s+", " ", v).strip()
    
    # Tokenize
    tokens = v.split(" ")
    
    # Remove empty tokens
    tokens = [t for t in tokens if t]
    
    # Keep only alphanumeric tokens (but plugin already fixed)
    tokens = [t for t in tokens if re.match(r"^[a-z0-9]+$", t)]
    
    # Remove duplicates
    tokens = list(set(tokens))
    
    # Sort tokens for consistency
    tokens = sorted(tokens)
    
    # Format tokens
    tokens = [("Plug-In" if t == "plugin" else t.title()) for t in tokens]
    
    return " ".join(tokens)

df_cleaned["fuel_types_clean"] = df_cleaned["fuel_types"].apply(normalize_fuel_type)


In [0]:
df_cleaned["fuel_types_clean"].groupby(df_cleaned["fuel_types_clean"]).count()



fuel_types_clean
Awd Petrol               2
Cng Petrol               1
Diesel                 105
Diesel Hybrid            1
Diesel Petrol           23
Electric                96
Electric Gas Hybrid      1
Electric Hybrid          1
Ev Petrol                1
Gas Hybrid               1
Hybrid                  77
Hybrid Petrol           23
Hybrid Plug-In           7
Hydrogen                 3
Petrol                 868
Name: fuel_types_clean, dtype: int64

In [0]:
# drop original column fuel_types
df_cleaned = df_cleaned.drop(columns=["fuel_types"])

In [0]:
# Rename the cleaned fuel_types column into fuel_types
df_cleaned = df_cleaned.rename(columns={"fuel_types_clean": "fuel_types"})

In [0]:
df_cleaned["fuel_types"].value_counts().head(20)

fuel_types
Petrol                 868
Diesel                 105
Electric                96
Hybrid                  77
Diesel Petrol           23
Hybrid Petrol           23
Hybrid Plug-In           7
Hydrogen                 3
Awd Petrol               2
Cng Petrol               1
Diesel Hybrid            1
Electric Gas Hybrid      1
Gas Hybrid               1
Ev Petrol                1
Electric Hybrid          1
Name: count, dtype: int64

#### 3.2.10 seats

In [0]:
df_cleaned['seats'].groupby(df_cleaned['seats']).count()

seats
1         3
12        1
2       134
2+2       5
20        1
212      1
215      1
26       2
27       1
29       1
3        14
4       160
5       689
6        20
7       146
78       1
8        29
9         1
Name: seats, dtype: int64

**Note**: The `seats` column contains heterogeneous formats describing the seating capacity of vehicles. Values include single integers (e.g., "5", "7"), compact notations such as "2+2", ranges with various separators ("7-8", "2-6"), and entries containing non-standard Unicode characters. The dataset also includes different vehicle categories (passenger cars, vans, minibuses, trucks), so
valid seat counts may range from 2 up to 20.

This cleaning step applies a general, rule-based normalization:

- normalize all separators to a standard form,
- extract all numeric values from each entry,
- if a single number is present, keep it,
- if multiple numbers are present, use the minimum (conservative rule),
- if the numeric range is unrealistically large (difference > 10), return NaN.

In [0]:
def normalize_seats(value):
    if pd.isna(value):
        return np.nan
    
    v = str(value).strip().lower()
    
    # Normalize separators (+, -, –, —, −, /)
    v = re.sub(r"[+\-–—−/]", "-", v)
    
    # Normalize spacing
    v = re.sub(r"\s+", "", v)
    
    # Extract all numbers
    nums = re.findall(r"\d+", v)
    
    if not nums:
        return np.nan
    
    nums = list(map(int, nums))
    
    # Single value
    if len(nums) == 1:
        return nums[0]
    
    # Unrealistic range
    if max(nums) - min(nums) > 10:
        return np.nan
    
    # Conservative choice
    return min(nums)

df_cleaned["seats_clean"] = df_cleaned["seats"].apply(normalize_seats)

# Convert to int
df_cleaned["seats_clean"] = df_cleaned["seats_clean"].astype("Int64")


In [0]:
df_cleaned["seats_clean"].groupby(df_cleaned["seats_clean"]).count()

seats_clean
1       3
2     144
3      14
4     160
5     689
6      20
7     147
8      29
9       1
12      1
20      1
Name: seats_clean, dtype: Int64

In [0]:
# Drop the original seats column
df_cleaned = df_cleaned.drop(columns=["seats"])
# Rename the cleaned seats column into seats
df_cleaned = df_cleaned.rename(columns={"seats_clean": "seats"})

#### 3.2.11 torque

In [0]:
df_cleaned["torque"].groupby(df_cleaned["torque"]).count()

    

torque
1,050 - 1,200 Nm    1
1,064 Nm            1
1,355 Nm            1
1,400 Nm            2
1,424 Nm            2
                   ..
93 Nm               2
94 Nm               2
95 Nm               3
950 Nm              2
967 Nm              2
Name: torque, Length: 263, dtype: int64

**Note**: The `torque` column contains heterogeneous formats describing engine torque.
Values include single numeric entries (e.g., "400 Nm"), ranges ("1,050 - 1,200 Nm"), numbers with thousand separators ("1,424 Nm"), and entries containing additional text such as "@ rpm". Some entries contain no numeric information (e.g.,"Instant torque").

This cleaning step applies a general, rule-based normalization:

- remove thousand separators,
- extract all numeric values from each entry,
- if a single number is present, keep it,
- if multiple numbers are present, use the minimum (conservative rule),
- if no numeric value is present, return NaN.

The cleaned numeric torque value (in Newton-meters) is stored in the new column `torque_nm`.

In [0]:
def normalize_torque(value):
    if pd.isna(value):
        return np.nan
    
    v = str(value).lower()
    
    # Remove commas (thousands)
    v = v.replace(",", "")
    
    # Remove unit
    v = v.replace("nm", "")
    
    # Remove anything after '@'
    v = v.split("@")[0]
    
    # Extract numbers
    nums = re.findall(r"\d+", v)
    
    if not nums:
        return np.nan
    
    nums = list(map(int, nums))
    
    # Single value
    if len(nums) == 1:
        return nums[0]
    
    # Multiple values - conservative choice
    return min(nums)

df_cleaned["torque_nm"] = df_cleaned["torque"].apply(normalize_torque)


In [0]:
df_cleaned["torque_nm"].groupby(df_cleaned["torque_nm"]).count()

torque_nm
45       1
51       1
60       3
62       1
80       1
        ..
2500     2
2600     1
3500     1
10000    1
15590    5
Name: torque_nm, Length: 233, dtype: int64

In [0]:
# Drop the original torque column
df_cleaned = df_cleaned.drop(columns=["torque"])

In [0]:
df_cleaned["torque_nm"].describe()

count     1210.000000
mean       506.654545
std       1052.515266
min         45.000000
25%        250.000000
50%        370.000000
75%        560.000000
max      15590.000000
Name: torque_nm, dtype: float64

In [0]:
df_cleaned[df_cleaned["torque_nm"].isna()]

,make,model,engines,cars_prices,engine_cc,battery_kwh,acc_min,acc_max,acc_mean,acc_is_range,acc_is_instant,is_commercial,hp_min,hp_max,hp_mean,hp_is_range,total_speed_kmh,fuel_types,seats,torque_nm


#### 3.2.12 cars_prices (Target)

In [0]:
df_cleaned["cars_prices"].groupby(df_cleaned["cars_prices"]).count().sort_values(ascending=False)


cars_prices
$35,000              36
$30,000              31
$50,000              31
$45,000              28
$55,000              28
                     ..
$29,200               1
$29,000  $43,000     1
$29,000 - $36,000     1
$29,000 - $32,000     1
 33,000              1
Name: cars_prices, Length: 534, dtype: int64

In [0]:
pattern_eur = r"[€\u0080]"
df_prices_euro = df_cleaned[df_cleaned["cars_prices"].astype(str).str.contains(pattern_eur, regex=True)]
df_prices_euro

,make,model,engines,cars_prices,engine_cc,battery_kwh,acc_min,acc_max,acc_mean,acc_is_range,acc_is_instant,is_commercial,hp_min,hp_max,hp_mean,hp_is_range,total_speed_kmh,fuel_types,seats,torque_nm
607,Toyota,C-HR+,1.8L / 2.0L Hybrid," 33,000",NaN,NaN,7.9,7.9,7.9,0,0,0,140.0,198.0,169.0,1,180.0,Hybrid,5,190


The `cars_prices` column contains heterogeneous price formats including:
- single USD values (e.g., "$35,000"),
- price ranges (e.g., "$29,000 - $36,000"),
- values with thousand separators ("," or "."),
- entries containing non-standard Unicode currency symbols (e.g., " 33,000").

Since only one record uses a euro-like symbol, no currency conversion is applied.
Instead, all currency symbols (USD or euro variants) are removed and the numeric value is interpreted directly.

This cleaning step applies a general, rule-based normalization:

- remove all currency symbols (including non-standard euro variants),
- remove thousand separators and spaces,
- normalize hyphens used in price ranges,
- extract all numeric values from each entry,
- if a single number is present, keep it,
- if multiple numbers are present (price range), use the minimum,
- return a clean numeric price as a float.

The cleaned target variable is stored in the new column `price_usd`.

In [0]:
def normalize_price(value):
    if pd.isna(value):
        return np.nan
    
    v = str(value).strip().lower()
    
    # Remove currency symbols
    v = re.sub(r"[\$€\u0080]", "", v)
    
    # Remove concept / invalid entries
    if "concept" in v or "n/a" in v:
        return np.nan
    
    # Normalize hyphens (ranges)
    v = re.sub(r"[–—−]", "-", v)
    
    # Remove spaces
    v = v.replace(" ", "")
    
    # remove thousand separators
    v = v.replace(",", "")
    
    # Extract full numbers (now safe)
    nums = re.findall(r"\d+", v)
    
    if not nums:
        return np.nan
    
    nums = list(map(int, nums))
    
    return float(min(nums))

df_cleaned["price_usd"] = df_cleaned["cars_prices"].apply(normalize_price)


In [0]:
df_cleaned["price_usd"].groupby(df_cleaned["price_usd"]).count().sort_values(ascending=False)

price_usd
35000.0       47
25000.0       44
30000.0       42
50000.0       35
40000.0       35
              ..
51570.0        1
51500.0        1
51095.0        1
50995.0        1
18000000.0     1
Name: price_usd, Length: 451, dtype: int64

In [0]:
df_cleaned.sort_values("price_usd", ascending=False).head(10)

,make,model,engines,cars_prices,engine_cc,battery_kwh,acc_min,acc_max,acc_mean,acc_is_range,acc_is_instant,is_commercial,hp_min,hp_max,hp_mean,hp_is_range,total_speed_kmh,fuel_types,seats,torque_nm,price_usd
278,Bugatti,La Voiture Noire,8.0L Quad-Turbo W16,"$18,000,000",7993.0,NaN,2.5,2.5,2.5,0,0,0,1500.0,1500.0,1500.0,0,420.0,Petrol,2,1600,18000000.0
277,Bugatti,Centodieci,8.0L Quad-Turbo W16,"$9,000,000",7993.0,NaN,2.4,2.4,2.4,0,0,0,1600.0,1600.0,1600.0,0,380.0,Petrol,2,1600,9000000.0
276,Bugatti,Divo,8.0L Quad-Turbo W16,"$5,800,000",7993.0,NaN,2.4,2.4,2.4,0,0,0,1500.0,1500.0,1500.0,0,380.0,Petrol,2,1600,5800000.0
280,Bugatti,Mistral,8.0L Quad-Turbo W16,"$5,000,000",7993.0,NaN,2.4,2.4,2.4,0,0,0,1600.0,1600.0,1600.0,0,420.0,Petrol,2,1600,5000000.0
603,Mazda,787B (Race Car),2.6L Wankel Rotary Engine,"$5,000,000 - $7,000,000",2620.0,NaN,3.5,3.5,3.5,0,0,0,700.0,700.0,700.0,0,320.0,Petrol,1,500,5000000.0
279,Bugatti,Bolide,8.0L Quad-Turbo W16,"$4,500,000",7993.0,NaN,2.2,2.2,2.2,0,0,0,1850.0,1850.0,1850.0,0,500.0,Petrol,2,1600,4500000.0
617,Lamborghini,VENENO ROADSTER,V12,"$4,500,000",6498.0,NaN,2.9,2.9,2.9,0,0,0,750.0,750.0,750.0,0,356.0,Petrol,2,690,4500000.0
273,Bugatti,Chiron Super Sport,8.0L Quad-Turbo W16,"$3,500,000",7993.0,NaN,2.4,2.4,2.4,0,0,0,1600.0,1600.0,1600.0,0,490.0,Petrol,2,1600,3500000.0
272,Bugatti,Chiron Pur Sport,8.0L Quad-Turbo W16,"$3,300,000",7993.0,NaN,2.3,2.3,2.3,0,0,0,1500.0,1500.0,1500.0,0,350.0,Petrol,2,1600,3300000.0
275,Bugatti,Chiron Noire,8.0L Quad-Turbo W16,"$3,300,000",7993.0,NaN,2.5,2.5,2.5,0,0,0,1500.0,1500.0,1500.0,0,420.0,Petrol,2,1600,3300000.0


In [0]:
# drop the original price column
df_cleaned = df_cleaned.drop(columns=["cars_prices"])

In [0]:
df_cleaned.head(30)

,make,model,engines,engine_cc,battery_kwh,acc_min,acc_max,acc_mean,acc_is_range,acc_is_instant,is_commercial,hp_min,hp_max,hp_mean,hp_is_range,total_speed_kmh,fuel_types,seats,torque_nm,price_usd
0,Porsche,911 Carrera S Cabriolet,3.0L Twin-Turbo Flat-6,2981.0,NaN,3.9,3.9,3.9,0,0,0,443.0,443.0,443.0,0,304.0,Petrol,4,530,129900.0
1,Porsche,911 Carrera 4,3.0L Twin-Turbo Flat-6,2981.0,NaN,4.4,4.4,4.4,0,0,0,379.0,379.0,379.0,0,293.0,Petrol,4,450,113300.0
2,Porsche,911 Carrera 4 GTS,3.0L Twin-Turbo Flat-6,2981.0,NaN,3.3,3.3,3.3,0,0,0,473.0,473.0,473.0,0,311.0,Petrol,4,570,137000.0
3,Porsche,911 Targa 4 GTS,3.0L Twin-Turbo Flat-6,2981.0,NaN,3.5,3.5,3.5,0,0,0,473.0,473.0,473.0,0,307.0,Petrol,4,570,157300.0
4,Porsche,911 Turbo Cabriolet,3.8L Twin-Turbo Flat-6,3745.0,NaN,2.9,2.9,2.9,0,0,0,572.0,572.0,572.0,0,320.0,Petrol,4,750,184100.0
5,Porsche,911 Turbo S Coupe,3.8L Twin-Turbo Flat-6,3745.0,NaN,2.7,2.7,2.7,0,0,0,640.0,640.0,640.0,0,330.0,Petrol,4,800,207000.0
6,Porsche,911 GT3 Touring,4.0L Naturally Aspirated Flat-6,3996.0,NaN,3.9,3.9,3.9,0,0,0,502.0,502.0,502.0,0,318.0,Petrol,2,470,170000.0
7,Porsche,911 GT3 RS Clubsport,4.0L Naturally Aspirated Flat-6,3996.0,NaN,3.2,3.2,3.2,0,0,0,525.0,525.0,525.0,0,296.0,Petrol,2,465,223800.0
8,Porsche,911 Dakar Limited Edition,3.0L Twin-Turbo Flat-6,2981.0,NaN,3.4,3.4,3.4,0,0,0,473.0,473.0,473.0,0,240.0,Petrol,4,570,222000.0
9,Porsche,911 Carrera GTS Lightweight,3.0L Twin-Turbo Flat-6,2981.0,NaN,3.2,3.2,3.2,0,0,0,473.0,473.0,473.0,0,311.0,Petrol,4,570,145300.0


In [0]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1210 entries, 0 to 1217
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   make             1210 non-null   object 
 1   model            1210 non-null   object 
 2   engines          1210 non-null   object 
 3   engine_cc        1062 non-null   float64
 4   battery_kwh      104 non-null    float64
 5   acc_min          1205 non-null   float64
 6   acc_max          1205 non-null   float64
 7   acc_mean         1205 non-null   float64
 8   acc_is_range     1210 non-null   int64  
 9   acc_is_instant   1210 non-null   int64  
 10  is_commercial    1210 non-null   int64  
 11  hp_min           1210 non-null   float64
 12  hp_max           1210 non-null   float64
 13  hp_mean          1210 non-null   float64
 14  hp_is_range      1210 non-null   int64  
 15  total_speed_kmh  1210 non-null   float64
 16  fuel_types       1210 non-null   object 
 17  seats            12

In [0]:
# Rename columns
df_cleaned = df_cleaned.rename(columns={

    # engine / powertrain
    "engines": "engine_description",
    "engine_cc": "engine_displacement_cc",
    "battery_kwh": "battery_capacity_kwh",

    # acceleration (0-100)
    "acc_min": "acc_0_100_min",
    "acc_max": "acc_0_100_max",
    "acc_mean": "acc_0_100_mean",
    "acc_is_range": "acc_0_100_is_range",
    "acc_is_instant": "acc_0_100_is_instant",

    # horsepower
    "hp_min": "horsepower_min",
    "hp_max": "horsepower_max",
    "hp_mean": "horsepower_mean",
    "hp_is_range": "horsepower_is_range",

    # speed
    "total_speed_kmh": "top_speed_kmh",

    # misc
    "fuel_types": "fuel_type"
})

In [0]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1210 entries, 0 to 1217
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   make                    1210 non-null   object 
 1   model                   1210 non-null   object 
 2   engine_description      1210 non-null   object 
 3   engine_displacement_cc  1062 non-null   float64
 4   battery_capacity_kwh    104 non-null    float64
 5   acc_0_100_min           1205 non-null   float64
 6   acc_0_100_max           1205 non-null   float64
 7   acc_0_100_mean          1205 non-null   float64
 8   acc_0_100_is_range      1210 non-null   int64  
 9   acc_0_100_is_instant    1210 non-null   int64  
 10  is_commercial           1210 non-null   int64  
 11  horsepower_min          1210 non-null   float64
 12  horsepower_max          1210 non-null   float64
 13  horsepower_mean         1210 non-null   float64
 14  horsepower_is_range     1210 non-null   int64

## 4. Export the Cleaned Dataframe into a Delta Table

In [0]:
# Export the cleaned dataset to Delta format in Unity Catalog

# Define the target table
target_catalog = "workspace"
target_schema = "caraiproject"
target_table = "cleaned_cars_data"
full_table_name = f"{target_catalog}.{target_schema}.{target_table}"

print(f"\nTarget table: {full_table_name}")

# Add export timestamp
df_cleaned['cleaning_timestamp'] = datetime.now()

# Convert pandas DataFrame to Spark DataFrame for Delta write
spark_df = spark.createDataFrame(df_cleaned)

# Write to Delta table
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(full_table_name)

# Add table comment and properties
spark.sql(f"""
    COMMENT ON TABLE {full_table_name} IS 
    'Cleaned cars dataset with normalized and standardized values. All columns have been processed to extract numeric values, standardize formats, and handle missing data. Ready for feature engineering and modeling. Processing framework: pandas.'
""")

spark.sql(f"""
    ALTER TABLE {full_table_name} SET TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'project' = 'CarAIProject',
        'pipeline_stage' = 'cleaned',
        'data_classification' = 'internal',
        'processing_framework' = 'pandas',
        'source_table' = 'workspace.caraiproject.raw_cars_data_gathered'
    )
""")

# Verify the table
final_row_count = spark.table(full_table_name).count()


print("EXPORT COMPLETED SUCCESSFULLY")
print(f"\nCleaned dataset exported to: {full_table_name}")
print(f"Processing: pandas DataFrame -> Spark DataFrame -> Delta Table")
print(f"Rows exported: {final_row_count:,}")
print(f"Ready for feature engineering and modeling pipeline")


Target table: workspace.caraiproject.cleaned_cars_data
EXPORT COMPLETED SUCCESSFULLY

Cleaned dataset exported to: workspace.caraiproject.cleaned_cars_data
Processing: pandas DataFrame -> Spark DataFrame -> Delta Table
Rows exported: 1,210
Ready for feature engineering and modeling pipeline
